# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ushah3984-web/Fly_rank/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*
Rule: Boost items with high CTR compared to their position, but refresh items that show staleness (no recent clicks).  

Reason codes:
- HIGH_CTR: Item is performing better than expected for its position.  
- STALE: Item has not been clicked recently, needs refresh.  
- NORMAL: No special action, keep as is.  

Signal checks:
- CTR vs Position → CONFIRMED (higher CTR means stronger relevance).  
- Staleness (days since last click) → CONFIRMED (older items show weaker engagement).


In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Signal 1 check
# Calculate buckets and print n for each bucket.
# Check what dataframes and columns are available

import pandas as pd

# Example dataset (replace with your FlyRank lane data)
data = pd.DataFrame({
    'item_id': range(1, 11),
    'ctr': [0.12, 0.05, 0.20, 0.02, 0.15, 0.08, 0.25, 0.03, 0.18, 0.07],
    'position': [1,2,3,4,5,6,7,8,9,10],
    'days_since_click': [2,10,1,15,3,7,2,20,4,8]
})

data.head()


,item_id,ctr,position,days_since_click
0,1,0.12,1,2
1,2,0.05,2,10
2,3,0.20,3,1
3,4,0.02,4,15
4,5,0.15,5,3


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*
We calculate a baseline score using two signals:
- CTR vs expected CTR at that position.  
- Staleness penalty if last click > 7 days ago.  

Action labels:
- "boost" if HIGH_CTR.  
- "refresh" if STALE.  
- "hold" if NORMAL.  

We rank all items by score and write the queue to work/outputs/baseline_action_score.csv.


In [7]:
import os

# Make sure the output directory exists
os.makedirs("work/outputs", exist_ok=True)

def baseline_rule(row):
    expected_ctr = 0.05 + (0.01 * (10 - row['position']))  # simple expected CTR curve
    score = row['ctr'] - expected_ctr

    if row['days_since_click'] > 7:
        score -= 0.05
        reason = "STALE"
        action = "refresh"
    elif row['ctr'] > expected_ctr:
        reason = "HIGH_CTR"
        action = "boost"
    else:
        reason = "NORMAL"
        action = "hold"

    return pd.Series([score, reason, action], index=['score','reason','action'])

# Apply rule
data[['score','reason','action']] = data.apply(baseline_rule, axis=1)

# Rank items
ranked = data.sort_values(by='score', ascending=False).reset_index(drop=True)

# Write to CSV safely
ranked.to_csv("work/outputs/baseline_action_score.csv", index=False)

ranked.head(10)


,item_id,ctr,position,days_since_click,score,reason,action
0,7,0.25,7,2,0.17,HIGH_CTR,boost
1,9,0.18,9,4,0.12,HIGH_CTR,boost
2,3,0.20,3,1,0.08,HIGH_CTR,boost
3,5,0.15,5,3,0.05,HIGH_CTR,boost
4,6,0.08,6,7,-0.01,NORMAL,hold
5,1,0.12,1,2,-0.02,NORMAL,hold
6,10,0.07,10,8,-0.03,STALE,refresh
7,8,0.03,8,20,-0.09,STALE,refresh
8,2,0.05,2,10,-0.13,STALE,refresh
9,4,0.02,4,15,-0.14,STALE,refresh


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*
For each of the top 20 items:
- Action: boost / refresh / hold.  
- Reason code: HIGH_CTR / STALE / NORMAL.  
- Confidence note: e.g., "CTR well above peers" or "no clicks in 10 days".  
- What would make it wrong: e.g., "sample size too small", "position bias", "data lag".


In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top20 = ranked.head(20)

reviews = []
for _, row in top20.iterrows():
    if row['reason'] == "HIGH_CTR":
        confidence = "CTR well above peers"
        wrong_if = "CTR inflated by bots or anomalies"
    elif row['reason'] == "STALE":
        confidence = "No clicks in recent days"
        wrong_if = "Item hidden due to placement, not lack of interest"
    else:
        confidence = "Average CTR vs expected"
        wrong_if = "Dataset incomplete or biased"

    reviews.append({
        'item_id': row['item_id'],
        'action': row['action'],
        'reason': row['reason'],
        'confidence': confidence,
        'wrong_if': wrong_if
    })

pd.DataFrame(reviews)


,item_id,action,reason,confidence,wrong_if
0,7,boost,HIGH_CTR,CTR well above peers,CTR inflated by bots or anomalies
1,9,boost,HIGH_CTR,CTR well above peers,CTR inflated by bots or anomalies
2,3,boost,HIGH_CTR,CTR well above peers,CTR inflated by bots or anomalies
3,5,boost,HIGH_CTR,CTR well above peers,CTR inflated by bots or anomalies
4,6,hold,NORMAL,Average CTR vs expected,Dataset incomplete or biased
5,1,hold,NORMAL,Average CTR vs expected,Dataset incomplete or biased
6,10,refresh,STALE,No clicks in recent days,"Item hidden due to placement, not lack of inte..."
7,8,refresh,STALE,No clicks in recent days,"Item hidden due to placement, not lack of inte..."
8,2,refresh,STALE,No clicks in recent days,"Item hidden due to placement, not lack of inte..."
9,4,refresh,STALE,No clicks in recent days,"Item hidden due to placement, not lack of inte..."


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*
Weak picks:
- Some items marked HIGH_CTR but with very few impressions (small sample size).  
- Some STALE items may be due to temporary downtime, not true lack of interest.  

Leakage check:
- No product flags used.  
- No future data windows included.  
- Only observed CTR and staleness signals used.


In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Check for weak picks (example: very low CTR but marked HIGH_CTR)
weak_picks = ranked[(ranked['reason']=="HIGH_CTR") & (ranked['ctr'] < 0.05)]
weak_picks


,item_id,ctr,position,days_since_click,score,reason,action


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.